# Explore the National-Scale Environmental Pressure Indicators Dataset

## Introduction
This notebook provides an exploratory data analysis (EDA) of the "National-Scale Environmental Pressure Indicators from 1990 to 2050" dataset. This dataset contains historical (1990–2020) and forecasted (2021–2050) national-level data for key environmental and socioeconomic indicators, including Municipal Solid Waste (MSW) generation, greenhouse gas emissions (CO₂, CH₄, N₂O), GDP per capita (PPP), and population for 43 countries.

Learn more:
- Data Package doi: [10.71728/senscience.k2f7-p5v9](https://doi.org/10.71728/senscience.k2f7-p5v9)

This notebook will guide you through loading, exploring, and visualizing the data to uncover key trends and insights.

### Install and import required libraries

In [ ]:
# Install mlcroissant and the analysis dependencies
!pip install mlcroissant==1.0.22
!pip install seaborn==0.13.2 plotly==6.1.2 nbformat==5.10.4 jinja2==3.1.6

In [ ]:
import sys
import tempfile
import urllib.request

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from IPython.display import Markdown, display

## 1. Data Loading
We load the dataset with `mlcroissant` from its FAIR² metadata, `fair2.json`, at the Data Package URL.

In [ ]:
PATCH_URL = 'https://raw.githubusercontent.com/senscience/notebooks/main/tools/patch_fair2.py'
patch_folder = tempfile.mkdtemp()
urllib.request.urlretrieve(PATCH_URL, f'{patch_folder}/patch_fair2.py')
sys.path.insert(0, patch_folder)
from patch_fair2 import load_dataset

url = 'https://sen.science/doi/10.71728/senscience.k2f7-p5v9/fair2.json'
dataset = load_dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}\n\n{metadata.description}")

In [ ]:
record_sets = {rs.name: rs for rs in metadata.record_sets}
display(pd.DataFrame([{'Record set': name, 'Fields': len(rs.fields), 'Description': rs.description}
                      for name, rs in record_sets.items()])
        .style.hide(axis='index').set_caption('Record sets'))

Load both record sets as DataFrames. `mlcroissant` names the columns by field `@id` and returns the values as bytes, and every field is declared as text in the metadata: the columns get short names, and the numbers are converted. The `Year` of the forecast is a year (1990–2020, the historical values again) or a scenario and a year, such as `SSP2_2035`, for the five Shared Socioeconomic Pathways (SSP1–SSP5): it is split into a scenario and a year.

In [ ]:
COLUMNS = {
    'Country/Region Name': 'Country', 'Year': 'Year', 'Population': 'Population',
    'GDP/capita (PPP, current international $)': 'GDP per capita (PPP $)',
    'MSW generation (t/year)': 'MSW (t/year)',
    'CO₂ emissions (t CO₂-eq/year)': 'CO₂ (t CO₂-eq/year)',
    'CH₄ emissions (t CO₂-eq/year)': 'CH₄ (t CO₂-eq/year)',
    'N₂O emissions (t CO₂-eq/year)': 'N₂O (t CO₂-eq/year)',
    'CH₄ emissions (t CO₂-we/year)': 'CH₄ (t CO₂-we/year)',
}

def load(name):
    record_set = record_sets[name]
    df = pd.DataFrame(dataset.records(record_set=record_set.id))
    df = df.rename(columns={field.id: COLUMNS[field.name] for field in record_set.fields})
    df = df.apply(lambda column: column.map(lambda v: v.decode('utf-8') if isinstance(v, bytes) else v))
    numbers = [column for column in df.columns if column not in ('Country', 'Year')]
    df[numbers] = df[numbers].apply(pd.to_numeric, errors='coerce')
    return df

historical = load('Historical')
historical['Year'] = historical['Year'].astype(int)

forecast_all = load('Forecast')
forecast_all['Scenario'] = forecast_all['Year'].str.extract(r'^(SSP\d)_', expand=False).fillna('Historical')
forecast_all['Year'] = forecast_all['Year'].str.extract(r'(\d{4})$', expand=False).astype(int)
forecast = forecast_all[forecast_all['Scenario'] != 'Historical'].copy()

# The years 1990-2020 of the forecast record set repeat the historical record set
repeated = forecast_all[forecast_all['Scenario'] == 'Historical'].merge(historical, on=['Country', 'Year'], suffixes=('', '_h'))
identical = all(np.allclose(repeated[c], repeated[f'{c}_h'], equal_nan=True) for c in ('Population', 'MSW (t/year)', 'CO₂ (t CO₂-eq/year)'))
print(f'The 1990-2020 rows of the forecast record set are identical to the historical record set: {identical}')

SCENARIOS = sorted(forecast['Scenario'].unique())
display(pd.DataFrame({
    'Rows': [len(historical), len(forecast)],
    'Countries': [historical['Country'].nunique(), forecast['Country'].nunique()],
    'Years': [f"{historical['Year'].min()}–{historical['Year'].max()}", f"{forecast['Year'].min()}–{forecast['Year'].max()}"],
    'Scenarios': ['—', ', '.join(SCENARIOS)],
}, index=['Historical', 'Forecast (scenarios)']).style.format({'Rows': '{:,}'}).set_caption('Loaded data'))

## 2. Data Overview
### 2.1 Missing values

In [ ]:
missing = pd.DataFrame({'Historical': historical.isna().sum(), 'Forecast (all rows)': forecast_all.isna().sum()}).fillna(0).astype(int)
missing = missing[missing.sum(axis=1) > 0]
display(missing.style.format('{:,}').set_caption('Missing values (columns with any)'))

The data is complete, except for the CH₄ emissions in t CO₂-we/year, a column of the forecast record set only, which is empty for the 1990–2020 rows. The analysis uses the CH₄ emissions in t CO₂-eq/year, available everywhere.

### 2.2 Totals in 2020 and 2050
The sum over the 43 countries, in 2020 and in 2050 for each scenario, with the change from 2020.

In [ ]:
TOTALS = ['Population', 'MSW (t/year)', 'CO₂ (t CO₂-eq/year)', 'CH₄ (t CO₂-eq/year)', 'N₂O (t CO₂-eq/year)']
totals = pd.concat([
    historical[historical['Year'] == 2020][TOTALS].sum().rename('2020 (historical)').to_frame().T,
    forecast[forecast['Year'] == 2050].groupby('Scenario')[TOTALS].sum().rename(index=lambda s: f'2050 {s}'),
]) / 1e6
totals.columns = ['Population (millions)', 'MSW (Mt/year)', 'CO₂ (Mt CO₂-eq)', 'CH₄ (Mt CO₂-eq)', 'N₂O (Mt CO₂-eq)']
change = (totals / totals.iloc[0] - 1) * 100
display(totals.style.format('{:,.0f}').set_caption('Totals of the 43 countries'))
display(change.iloc[1:].style.format('{:+.0f}%').background_gradient(cmap='Blues', axis=None)
        .set_caption('Change from 2020 to 2050'))

By 2050, the waste generated grows in every scenario, from about +45% (SSP3) to almost +90% (SSP5), and the population by 10–24%. The emissions grow by the same amount in every scenario: unlike the waste, their projections do not depend on the scenario.

### 2.3 Normalizing: waste per person
Countries differ in size by two orders of magnitude: dividing the waste by the population gives the waste generated per person and per day, comparable across countries.

In [ ]:
latest = historical[historical['Year'] == 2020].copy()
latest['MSW per person (kg/day)'] = latest['MSW (t/year)'] * 1000 / latest['Population'] / 365
per_person = latest.set_index('Country')[['MSW per person (kg/day)', 'GDP per capita (PPP $)', 'Population']].sort_values('MSW per person (kg/day)')
display(pd.concat([per_person.head(5), per_person.tail(5)])
        .style.format({'MSW per person (kg/day)': '{:.2f}', 'GDP per capita (PPP $)': '{:,.0f}', 'Population': '{:,.0f}'})
        .set_caption('Lowest and highest waste per person in 2020'))
rho = per_person['MSW per person (kg/day)'].corr(per_person['GDP per capita (PPP $)'], method='spearman')
print(f'Spearman correlation between waste per person and GDP per capita in 2020: {rho:.2f}')

## 3. Visualization
### 3.1 Waste generation by country in 2020
The total waste generated by each country in 2020, the last historical year, on a logarithmic scale. Hover over a country for its value.

In [ ]:
# The country names of the dataset follow the World Bank; Plotly needs ISO 3166-1 alpha-3 codes
ISO3 = {
    'Algeria': 'DZA', 'Argentina': 'ARG', 'Australia': 'AUS', 'Bangladesh': 'BGD', 'Brazil': 'BRA', 'Canada': 'CAN',
    'China': 'CHN', 'Colombia': 'COL', 'Congo, Dem. Rep.': 'COD', 'Egypt, Arab Rep.': 'EGY', 'Ethiopia': 'ETH',
    'France': 'FRA', 'Germany': 'DEU', 'India': 'IND', 'Indonesia': 'IDN', 'Iran, Islamic Rep.': 'IRN', 'Iraq': 'IRQ',
    'Italy': 'ITA', 'Japan': 'JPN', 'Korea, Rep.': 'KOR', 'Malaysia': 'MYS', 'Mexico': 'MEX', 'Morocco': 'MAR',
    'Netherlands': 'NLD', 'Nigeria': 'NGA', 'Pakistan': 'PAK', 'Peru': 'PER', 'Philippines': 'PHL', 'Poland': 'POL',
    'Russian Federation': 'RUS', 'Saudi Arabia': 'SAU', 'South Africa': 'ZAF', 'Spain': 'ESP', 'Taiwan, China': 'TWN',
    'Tanzania': 'TZA', 'Thailand': 'THA', 'Turkey': 'TUR', 'Uganda': 'UGA', 'Ukraine': 'UKR', 'United Kingdom': 'GBR',
    'United States': 'USA', 'Venezuela, RB': 'VEN', 'Vietnam': 'VNM',
}
unmapped = set(latest['Country']) - set(ISO3)
if unmapped:
    print('Countries without an ISO code, not shown on the map:', unmapped)

fig = go.Figure(go.Choropleth(
    locations=latest['Country'].map(ISO3), locationmode='ISO-3',
    z=np.log10(latest['MSW (t/year)']), colorscale='Blues',
    marker_line_color='white', marker_line_width=0.5,
    hovertext=latest['Country'] + '<br>' + latest['MSW (t/year)'].map(lambda v: f'{v / 1e6:,.1f} Mt/year')
              + '<br>' + latest['MSW per person (kg/day)'].map(lambda v: f'{v:.2f} kg per person per day'),
    hoverinfo='text', colorbar_title='log₁₀(MSW, t/year)',
))
fig.update_layout(title_text='Municipal solid waste generated in 2020 (log₁₀ scale)',
                  geo=dict(showframe=False, showcoastlines=True, projection_type='natural earth'),
                  margin=dict(l=0, r=0, t=50, b=0))
fig.show()

### 3.2 Waste generation trends by scenario
The historical waste generation of four large countries, and its projection in each of the five scenarios. The scenarios are plotted separately: they are alternative futures, not an uncertainty range around a single forecast.

In [ ]:
COUNTRIES = ['United States', 'China', 'India', 'Brazil']
SCENARIO_COLORS = dict(zip(SCENARIOS, ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4']))

fig, axes = plt.subplots(2, 2, figsize=(15, 9), sharex=True)
for ax, country in zip(axes.flat, COUNTRIES):
    past = historical[historical['Country'] == country]
    ax.plot(past['Year'], past['MSW (t/year)'] / 1e6, color='#0b0b0b', linewidth=2, label='Historical')
    last = past[past['Year'] == past['Year'].max()]
    for scenario in SCENARIOS:
        future = forecast[(forecast['Country'] == country) & (forecast['Scenario'] == scenario)]
        future = pd.concat([last, future])   # start each projection from the last historical year
        ax.plot(future['Year'], future['MSW (t/year)'] / 1e6, color=SCENARIO_COLORS[scenario], linewidth=1.8, label=scenario)
    ax.axvline(2020.5, color='#898781', linewidth=1, linestyle='--')
    ax.set(title=country, ylabel='MSW (Mt/year)')
    ax.grid(color='#e1e0d9', linewidth=0.8)
axes[0, 0].legend(fontsize=9)
for ax in axes[1]:
    ax.set_xlabel('Year')
plt.suptitle('Municipal solid waste: historical and projected by scenario')
plt.tight_layout()
plt.show()

growth = pd.DataFrame({
    country: forecast[(forecast['Country'] == country) & (forecast['Year'] == 2050)].set_index('Scenario')['MSW (t/year)']
    / historical[(historical['Country'] == country) & (historical['Year'] == 2020)]['MSW (t/year)'].iloc[0] * 100 - 100
    for country in COUNTRIES}).T
display(growth.style.format('{:+.0f}%').background_gradient(cmap='Blues', axis=None)
        .set_caption('Change in waste generation from 2020 to 2050, by scenario'))

### 3.3 Waste, income and emissions
Left: the waste generated per person and the GDP per capita of each country in 2020. Right: the correlations between the historical indicators (all countries and years), on a scale centred on zero.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6.5), gridspec_kw={'width_ratios': [1, 1.1]})

ax = axes[0]
ax.scatter(per_person['GDP per capita (PPP $)'], per_person['MSW per person (kg/day)'],
           s=np.sqrt(per_person['Population']) / 40, color='#2a78d6', alpha=0.7, edgecolor='white', linewidth=0.8)
for country in COUNTRIES + [per_person.index[0], per_person.index[-1]]:
    point = per_person.loc[country]
    ax.annotate(country, (point['GDP per capita (PPP $)'], point['MSW per person (kg/day)']),
                xytext=(5, 4), textcoords='offset points', fontsize=9, color='#52514e')
ax.set(xscale='log', xlabel='GDP per capita (PPP, current international $, log scale)',
       ylabel='Waste per person (kg/day)', title='Waste per person and income, 2020 (size: population)')
ax.grid(color='#e1e0d9', linewidth=0.8)

indicators = ['Population', 'GDP per capita (PPP $)', 'MSW (t/year)', 'CO₂ (t CO₂-eq/year)', 'CH₄ (t CO₂-eq/year)', 'N₂O (t CO₂-eq/year)']
corr_matrix = historical[indicators].corr()
corr_matrix.index = corr_matrix.columns = [name.split(' (')[0] for name in indicators]
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, center=0, fmt='.2f', linewidths=0.5, ax=axes[1])
axes[1].set_title('Correlation of the historical indicators')
plt.tight_layout()
plt.show()

## 4. Conclusion
This notebook explored the national-scale waste and emissions dataset of 43 countries: historical data from 1990 to 2020, and projections to 2050 under five Shared Socioeconomic Pathways (SSP1–SSP5).

- **Data**: complete, apart from one CH₄ column of the forecast; the forecast record set repeats the historical years, and its years carry the scenario (`SSP2_2035`).
- **Scenarios**: by 2050 the waste generated by the 43 countries grows by about 45% (SSP3) to 90% (SSP5); the emission projections grow by the same amount in every scenario. Among the four countries shown, India's waste grows the most, roughly doubling.
- **Waste per person** rises with income: from under 0.2 kg per person and per day in Bangladesh and Ethiopia to over 2 kg in the United States.
- **Correlations**: the total waste is strongly correlated with CH₄ and N₂O emissions and with the population, and more moderately with CO₂; GDP per capita is moderately correlated with CO₂ only. Totals are dominated by country size: per-person indicators, as above, separate size from income.

Further analyses can compare the scenarios for other countries, relate the emissions to the waste per person, or study the decoupling of waste generation from income over time.